In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import re
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/"

DATASET_ID = "produccion"
TABLE_ID= "Declarados_CONTICASA"

#FECHA_PERIODO= "2025-09-05"
#PROJECT_ID = "test-proyect-468615"
#BUCKET_NAME = "data_bucket_proy"
#FOLDER_PATH= "desgravamen_prestamos/"
#DATASET_ID = "db_test"
#TABLE_ID= "desgravamen_prestamos"


# SCRIPT COMPLETO

In [ ]:
### CLIENTES DE STORAGE Y BIGQUERY
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

schema_declarados = [
        bigquery.SchemaField("CERTIFICADO_BANCO ", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CANT_COBERTURAS ", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_INDIV", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMA_TOTAL", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("OBSERVACIONES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("POLIZA_Y_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_DECLARADO", bigquery.enums.SqlTypeNames.DATETIME)
    ]

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return


### LISTAR LOS ARCHIVOS QUE ESTAN DENTRO DEL BUCKET
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

### CICLO POR LA LISTA DE ARCHIVOS EXCEL
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    print(f" ------- CARGANDO ARCHIVO: {blob.name}")
    file = blob.download_as_string()
    df_declarados= pd.read_excel(BytesIO(file), sheet_name='Hoja1', dtype={'N° Certificado Banco': str} )

    ###### LIMPIEZA Y TRANSFORMACIONES

     #Seleccionar solo las primaras 7 columnas
    df_declarados = df_declarados[df_declarados.columns[:7]]

    # Colocar _ en los espacio de los nombres de las columnas
    df_declarados.columns = (df_declarados.columns.str.strip()
                                                    .str.upper()  # opcional: todo en mayúsculas
                                                    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
                              )

    #Cambiar nombre a las columnas
    df_declarados= df_declarados.rename(columns={'N__CERTIFICADO_BANCO': 'CERTIFICADO_BANCO',
                                             'CANTIDAD_DE_COBERTURAS_A_RENOVAR': 'CANT_COBERTURAS',
                                             'P_LIZA_Y_CERTIF_': 'POLIZA_Y_CERTIFICADO'})

    #Extraer Fecha del nombre de archivo y guardarda en el campo "FECHA_DECLARADO"
    match = re.search(r"Declarado_(\d{2}-\d{4})", blob.name)
    if match:
      fecha_str = match.group(1)
      fecha_dt = datetime.strptime(fecha_str, "%m-%Y")
      df_declarados['FECHA_DECLARADO']= fecha_dt
    else:
      print(f"⚠️ Nombre de archivo inválido: {blob.name}")

    #Cambiar el tipo de datos de las columnas
    df_declarados["POLIZA_Y_CERTIFICADO"] = df_declarados["POLIZA_Y_CERTIFICADO"].astype(str)

    # Guardar tabla en BigQuery
    Guardar_en_BigQuery(df_declarados, DATASET_ID, TABLE_ID, schema_declarados)
    print(f"### EL ARCHIVO: {blob.name} SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###")



 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_01-2025_Declarado_02-2025.xlsx
----- Se ha creado la tabla Declarados_CONTICASA en el dataset produccion -----
### EL ARCHIVO: data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_01-2025_Declarado_02-2025.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###
 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_02-2025_Declarado_03-2025.xlsx
### EL ARCHIVO: data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_02-2025_Declarado_03-2025.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###
 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_03-2025_Declarado_04-2025.xlsx
### EL ARCHIVO: data_entries/REPORTES DE ERRORES/CONTICAS

--------------

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

In [ ]:
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    print(blob.name)
    #print(excel_file.sheet_names)

data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_01-2025_Declarado_02-2025.xlsx
data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_02-2025_Declarado_03-2025.xlsx
data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_03-2025_Declarado_04-2025.xlsx
data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_04-2025_Declarado_05-2025.xlsx
data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_05-2025_Declarado_06-2025.xlsx
data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_06-2025_Declarado_07-2025.xlsx
data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_07-2025_Declarado_08-2025.xlsx
data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación 

In [ ]:
#blob_consuer = bucket.blob('desgravamen_prestamos/desgravamen - consuer.csv')
#file = blob_consuer.download_as_string()
#df_desgravamen= pd.read_csv(BytesIO(file), dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str, 'IDELOTE': str, 'CODIGO ERROR': str})

In [ ]:
print(blobs_excels[1].name)
file = blobs_excels[1].download_as_string()
df_declarados= pd.read_excel(BytesIO(file), sheet_name='Hoja1', dtype={'N° Certificado Banco': str} )

data_entries/REPORTES DE ERRORES/CONTICASA/DECLARACION-CONTICASA/Renovación de Coberturas _Conticasa_02-2025_Declarado_03-2025.xlsx


In [ ]:
df_declarados.head(3)

,N° Certificado Banco,Cantidad de Coberturas a Renovar,MONEDA,PRIMA INDIV,Prima Total,OBSERVACIONES,Póliza y Certif.,cartera,cantidad-renovar,prima total-renovar,Unnamed: 10,Unnamed: 11,Unnamed: 12,vigencia,Unnamed: 14,fecha-vcto,cantidad-renovar.1,monto total
0,00110355504000287632,1,PEN,149.95,149.95,Generar cobertura,1403-500198-35538,2025-02-06,NaN,NaN,2025-02-06,NaN,NaN,renueva-feb a marzo,NaN,16/04/2025,NaN,NaN
1,00110355514000215437,1,PEN,100.07,100.07,Generar cobertura,1403-500198-12079,2025-03-03,NaN,NaN,2025-03-03,NaN,NaN,renueva-feb a marzo,NaN,09/04/2025,NaN,NaN
2,00110328244000174911,1,PEN,80.99,80.99,Generar cobertura,1403-500198-5230,2025-03-06,NaN,NaN,2025-03-06,NaN,NaN,renueva-feb a marzo,NaN,08/04/2025,NaN,NaN


In [ ]:
#Eliminar columnas sin nombre
df_declarados = df_declarados[df_declarados.columns[:7]]

df_declarados.columns = (
    df_declarados.columns
    .str.strip()  # quitar espacios al inicio/fin
    .str.upper()  # opcional: todo en mayúsculas
    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)
#Cambiar nombre a las columnas
df_declarados= df_declarados.rename(columns={'N__CERTIFICADO_BANCO': 'CERTIFICADO_BANCO',
                                             'CANTIDAD_DE_COBERTURAS_A_RENOVAR': 'CANT_COBERTURAS',
                                             'P_LIZA_Y_CERTIF_': 'POLIZA_Y_CERTIFICADO'})


In [ ]:
match = re.search(r"Declarado_(\d{2}-\d{4})", blob.name)
if match:
  fecha_str = match.group(1)
  fecha_dt = datetime.strptime(fecha_str, "%m-%Y")
  df_declarados['FECHA_DECLARADO']= fecha_dt
else:
  print(f"⚠️ Nombre de archivo inválido: {blob.name}")

In [ ]:
df_declarados["POLIZA_Y_CERTIFICADO"] = df_declarados["POLIZA_Y_CERTIFICADO"].astype(str)

In [ ]:
df_declarados.head()

,CERTIFICADO_BANCO,CANT_COBERTURAS,MONEDA,PRIMA_INDIV,PRIMA_TOTAL,OBSERVACIONES,POLIZA_Y_CERTIFICADO,FECHA_DECLARADO
0,00110355504000287632,1,PEN,149.95,149.95,Generar cobertura,1403-500198-35538,2025-01-01
1,00110355514000215437,1,PEN,100.07,100.07,Generar cobertura,1403-500198-12079,2025-01-01
2,00110328244000174911,1,PEN,80.99,80.99,Generar cobertura,1403-500198-5230,2025-01-01
3,00110355544000215445,1,PEN,168.76,168.76,Generar cobertura,1403-500198-12081,2025-01-01
4,00110355544000214945,1,PEN,98.83,98.83,Generar cobertura,1403-500198-11640,2025-01-01


In [ ]:
df_declarados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46803 entries, 0 to 46802
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   CERTIFICADO_BANCO     46803 non-null  object        
 1   CANT_COBERTURAS       46803 non-null  int64         
 2   MONEDA                46803 non-null  object        
 3   PRIMA_INDIV           46803 non-null  float64       
 4   PRIMA_TOTAL           46803 non-null  float64       
 5   OBSERVACIONES         46803 non-null  object        
 6   POLIZA_Y_CERTIFICADO  46803 non-null  object        
 7   FECHA_DECLARADO       46803 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(1), object(4)
memory usage: 2.9+ MB


In [ ]:
#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

In [ ]:
schema_declarados = [
        bigquery.SchemaField("CERTIFICADO_BANCO ", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CANT_COBERTURAS ", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_INDIV", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMA_TOTAL", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("OBSERVACIONES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("POLIZA_Y_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_DECLARADO", bigquery.enums.SqlTypeNames.DATETIME)
    ]
Guardar_en_BigQuery(df_declarados, DATASET_ID, TABLE_ID, schema_declarados)

----- Se ha creado la tabla ERRORES_CONTICASA en el dataset produccion -----
